In [30]:
import pandas as pd
import numpy as np
from enum import Enum
from dataclasses import dataclass
from typing import List
from sklearn.linear_model import LinearRegression
import warnings

# Suppress sklearn warnings for cleaner console output
warnings.filterwarnings('ignore')

# ==========================================
# 1. ENUMS & CONFIGURATIONS
# ==========================================

class SalesMetricSheet(Enum):
    """
    Defines which metric to forecast by mapping it to the exact Sheet Name in the Excel file.
    """
    NET_PRIMARY = 'Primary'
    SECONDARY = 'Secondary'

class TrainingPeriod(Enum):
    """
    Defines the historical months used to train the linear model.
    """
    JAN_TO_MAY = ['2026-01-26', '2026-02-26', '2026-03-26', '2026-04-26', '2026-05-26']

@dataclass
class ForecastConfig:
    """
    Holds all configuration parameters to avoid hard-coding.
    """
    input_file_path: str
    metric_sheet: SalesMetricSheet
    training_period: TrainingPeriod
    forecast_months: List[str]
    sku_column_name: str = 'SKU Code'

# ==========================================
# 2. DATA PROCESSING FUNCTIONS
# ==========================================

def load_and_clean_data(config: ForecastConfig) -> pd.DataFrame:
    """
    Loads data directly from the specified sheet in the Excel file and performs basic cleaning.
    """
    # Read specifically from the configured sheet (Primary or Secondary)
    df = pd.read_excel(config.input_file_path, sheet_name=config.metric_sheet.value)

    # Excel automatically converts date headers to Datetime objects.
    # Convert them back to strings (YYYY-MM-DD) so they match our Enum configuration perfectly.
    df.columns = [col.strftime('%Y-%m-%d') if hasattr(col, 'strftime') else str(col) for col in df.columns]

    # Ensure we only process rows that have a valid SKU
    if config.sku_column_name in df.columns:
        df = df.dropna(subset=[config.sku_column_name])
        df = df[df[config.sku_column_name].astype(str).str.strip() != '']

    return df

def extract_time_series_for_sku(df: pd.DataFrame, sku: str, config: ForecastConfig) -> np.ndarray:
    """
    Locates the specific SKU and extracts its historical metric values as a numpy array.
    """
    # Fetch the exact row for the SKU
    sku_data = df[df[config.sku_column_name] == sku].iloc[0]

    # Target columns are explicitly the 5 dates (Jan-May) configured in the Enum
    target_columns = config.training_period.value

    # Extract the data and treat missing/invalid inputs (-) as 0 units
    raw_values = sku_data[target_columns].replace('-', 0).fillna(0).astype(float).values
    return raw_values

# ==========================================
# 3. FORECASTING FUNCTIONS
# ==========================================

def fit_and_forecast_linear_trend(historical_data: np.ndarray, steps_ahead: int, start_step_offset: int) -> List[float]:
    """
    Fits a localized Linear Regression trendline on the time index to project future unit sales.
    """
    n_points = len(historical_data)

    # Return flat zero if the historical array is entirely empty or zeroed out
    if n_points == 0 or np.all(historical_data == 0):
        return [0.0] * steps_ahead

    # Time indices representing Jan(1), Feb(2), Mar(3), Apr(4), May(5)
    X_train = np.arange(1, n_points + 1).reshape(-1, 1)
    y_train = historical_data

    model = LinearRegression()
    model.fit(X_train, y_train)

    # Calculate future time steps for June, July, August
    # Because May is index 5, June is natively index 6 (start_step_offset = 1)
    future_indices = np.arange(n_points + start_step_offset, n_points + start_step_offset + steps_ahead).reshape(-1, 1)

    predictions = model.predict(future_indices)

    # Floor negative forecasts to 0 since unit sales cannot drop below zero
    predictions = np.maximum(predictions, 0)

    return predictions.tolist()

# ==========================================
# 4. MAIN PIPELINE ORCHESTRATION
# ==========================================

def run_forecasting_pipeline(config: ForecastConfig) -> pd.DataFrame:
    """
    Loops through every SKU, executes the model individually, and builds a DataFrame of forecasts.
    """
    df = load_and_clean_data(config)
    skus = df[config.sku_column_name].unique()

    results = []

    # For a Jan-May training period (5 points), predicting June has an offset of 1 (immediate next step)
    start_step_offset = 1
    steps_ahead = len(config.forecast_months)

    for sku in skus:
        historical_data = extract_time_series_for_sku(df, sku, config)
        forecasts = fit_and_forecast_linear_trend(historical_data, steps_ahead, start_step_offset)

        # Build the final output mapping
        result_row = {config.sku_column_name: sku}
        for month_name, forecast_val in zip(config.forecast_months, forecasts):
            result_row[f"Forecast_{month_name}"] = round(forecast_val, 2)

        results.append(result_row)

    return pd.DataFrame(results)

# ==========================================
# 5. EXECUTION ENTRY POINT
# ==========================================

if __name__ == "__main__":

    # 🚨 Force pandas to display ALL rows and columns in the console output
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)

    # Path to your uploaded Excel file
    EXCEL_FILE_PATH = "ChilRun Raw.xlsx"
    FORECAST_MONTHS = ['June_2026', 'July_2026', 'August_2026']

    print(f"--- Running Jan-May Forecast for Primary & Secondary Sales ---")

    # 1. Execute PRIMARY Sales Forecast
    config_primary = ForecastConfig(
        input_file_path=EXCEL_FILE_PATH,
        metric_sheet=SalesMetricSheet.NET_PRIMARY,
        training_period=TrainingPeriod.JAN_TO_MAY,
        forecast_months=FORECAST_MONTHS
    )

    forecast_df_primary = run_forecasting_pipeline(config_primary)

    print("\n[PRIMARY SALES FORECAST - ALL SKUS]")
    print(forecast_df_primary) # Removed .head() to show everything

    # Optional: Save Primary forecast to CSV
    forecast_df_primary.to_csv("Forecast_Primary_JanToMay.csv", index=False)


    # 2. Execute SECONDARY Sales Forecast
    config_secondary = ForecastConfig(
        input_file_path=EXCEL_FILE_PATH,
        metric_sheet=SalesMetricSheet.SECONDARY,
        training_period=TrainingPeriod.JAN_TO_MAY,
        forecast_months=FORECAST_MONTHS
    )

    forecast_df_secondary = run_forecasting_pipeline(config_secondary)

    print("\n[SECONDARY SALES FORECAST - ALL SKUS]")
    print(forecast_df_secondary) # Removed .head() to show everything

    # Optional: Save Secondary forecast to CSV
    forecast_df_secondary.to_csv("Forecast_Secondary_JanToMay.csv", index=False)

--- Running Jan-May Forecast for Primary & Secondary Sales ---

[PRIMARY SALES FORECAST - ALL SKUS]
      SKU Code  Forecast_June_2026  Forecast_July_2026  Forecast_August_2026
0   F5006SM300               797.9               938.4                1078.9
1   F5006SM200              1267.8              1511.6                1755.4
2   F5005SM200               620.5               777.8                 935.1
3   F5006SM100               136.5               156.6                 176.7
4   F5005SM300                 0.0                 0.0                  40.6
5   F5005SM100               185.1               228.6                 272.1
6   F5002SM200              2392.1              2911.8                3431.5
7   F5002SM300              1535.1              1866.4                2197.7
8   F5002SM400              1496.5              1722.8                1949.1
9   F5001SM200              1286.2              1556.6                1827.0
10  F5001SM400               571.6               697.